In [ ]:
import os

import pandas as pd

# The committed dataset is de-identified (Name = P1, P2, ...). If the gitignored
# identity map is present locally, show real names instead. Anyone without it
# (i.e. anyone cloning the repo) automatically sees the aliases.
USE_NAMES = False

df = pd.read_csv("responses_anon.csv")

if USE_NAMES and os.path.exists("participant_map.csv"):
    name_by_alias = pd.read_csv("participant_map.csv").set_index("alias")["name"]
    df["Name"] = df["Name"].map(name_by_alias).fillna(df["Name"])

# One respondent answered the Q5 ranking on a reversed scale — flip it (1<->5).
rev = df["q5_reversed"].fillna(False).astype(bool)
for c in [c for c in df.columns if c.startswith("Q5.")]:
    df.loc[rev, c] = df.loc[rev, c].map(lambda v: str(6 - int(str(v).split()[0])))

df.head()

In [ ]:
import matplotlib.pyplot as plt

orders = {
    "Q1": ["very unrealistic", "unrealistic", "borderline", "realistic", "very realistic"],
    "Q2": ["<5%", "5-10%", "10-20%", "20-30%", "30+%"],
    "Q3": ["< annually", "annually", "monthly", "daily", "10x per day", "> 10x per day"],
    "Q4": ["Negligible", "Minor", "Moderate", "Large", "Very Large"],
}
labels = {"Q1": "realism", "Q2": "persuasion", "Q3": "frequency", "Q4": "impact"}
scenarios = {
    "S1": "S1: CEO relaxes safety standards",
    "S2": "S2: safety program deprioritized",
    "S3": "S3: merges vulnerable PR",
    "S4": "S4: security lead exfiltrates weights",
    "S5": "S5: analyst suppresses alerts",
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (q, order) in zip(axes.flat, orders.items()):
    counts = pd.DataFrame({c.split(".")[0]: df[c].value_counts().reindex(order, fill_value=0) for c in df.columns if "." + q + "." in c})
    counts.plot.bar(ax=ax, legend=False)
    ax.set_title(f"{q} {labels[q]} by scenario")
    ax.set_ylabel("responses")
handles = axes[0, 0].containers
fig.legend(handles, [scenarios[c] for c in counts.columns], loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout()
plt.show()

In [ ]:
expl = [c for c in df.columns if c.startswith("Provide a short explanation")]

for q, label in labels.items():
    print(f"===== {q} {label} =====")
    for s, ecol in zip(scenarios, expl):
        print(f"\n--- {scenarios[s]} ---")
        acol = next(c for c in df.columns if c.startswith(f"{s}.{q}."))
        for lvl in orders[q]:
            sub = df[df[acol] == lvl]
            if sub.empty:
                continue
            print(f"  [{lvl}]")
            for name, ans in zip(sub["Name"], sub[ecol]):
                print(f"    {name}: {ans}")
    print()

In [ ]:
import numpy as np

reviewers = df["Name"].tolist()
rng = np.random.default_rng(0)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (q, order) in zip(axes.flat, orders.items()):
    rmap = {lvl: i for i, lvl in enumerate(order)}
    M = pd.DataFrame({s: df[next(c for c in df.columns if c.startswith(f"{s}.{q}."))].map(rmap) for s in scenarios})
    for i in range(len(reviewers)):
        y = M.iloc[i].dropna().values
        ax.scatter(i + rng.uniform(-0.15, 0.15, len(y)), y, color="C0", alpha=0.6, s=35)
    ax.scatter(range(len(reviewers)), M.mean(axis=1), color="k", marker="_", s=500)
    ax.set_yticks(range(len(order)), order)
    ax.set_xticks(range(len(reviewers)), reviewers, rotation=45, ha="right")
    ax.set_title(f"{q} {labels[q]}")
    ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
r1 = {lvl: i for i, lvl in enumerate(orders["Q1"])}
r2 = {lvl: i for i, lvl in enumerate(orders["Q2"])}
rows = []
for s in scenarios:
    c1 = next(c for c in df.columns if c.startswith(f"{s}.Q1."))
    c2 = next(c for c in df.columns if c.startswith(f"{s}.Q2."))
    for name, a, b in zip(df["Name"], df[c1], df[c2]):
        rows.append((name, r1.get(a), r2.get(b)))
long = pd.DataFrame(rows, columns=["name", "q1", "q2"]).dropna()

def style(ax):
    ax.set_xticks(range(len(orders["Q1"])), orders["Q1"], rotation=45, ha="right")
    ax.set_yticks(range(len(orders["Q2"])), orders["Q2"])
    ax.set_xlabel("Q1 realism")
    ax.set_ylabel("Q2 persuasion")

fig, ax = plt.subplots(figsize=(7, 5))
j = lambda n: rng.uniform(-0.1, 0.1, n)
ax.scatter(long.q1 + j(len(long)), long.q2 + j(len(long)), alpha=0.5)
m, b = np.polyfit(long.q1, long.q2, 1)
xs = np.array([long.q1.min(), long.q1.max()])
ax.plot(xs, m * xs + b, "k--", label=f"slope={m:.2f}")
ax.legend()
style(ax)
ax.set_title("Q2 vs Q1 (overall)")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True, sharey=True)
for ax, name in zip(axes.flat, reviewers):
    sub = long[long.name == name]
    ax.scatter(sub.q1 + j(len(sub)), sub.q2 + j(len(sub)), alpha=0.7)
    ax.set_title(name)
    ax.grid(alpha=0.3)
for ax in axes.flat[len(reviewers):]:
    ax.axis("off")
for ax in axes.flat:
    style(ax)
fig.suptitle("Q2 vs Q1 per respondent")
fig.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (q, order) in zip(axes.flat, orders.items()):
    rmap = {lvl: i for i, lvl in enumerate(order)}
    d = pd.DataFrame([(s, v) for s in scenarios for v in df[next(c for c in df.columns if c.startswith(f"{s}.{q}."))].map(rmap).dropna()], columns=["scenario", "val"])
    sns.violinplot(ax=ax, data=d, x="val", y="scenario", order=scenarios, orient="h", cut=0, density_norm="count", bw_adjust=0.5, inner=None, color="C0", linewidth=0)
    for coll in ax.collections:
        coll.set_alpha(0.3)
    med = d.groupby("scenario")["val"].median().reindex(scenarios)
    ax.scatter(med.values, range(len(scenarios)), marker="|", s=400, color="k", zorder=3)
    ax.set_xticks(range(len(order)), order, rotation=45, ha="right")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(f"{q} {labels[q]}")
fig.tight_layout()
plt.show()

In [ ]:
for q, order in orders.items():
    print(f"===== {q} {labels[q]} =====")
    mat = pd.DataFrame({s: df[next(c for c in df.columns if c.startswith(f"{s}.{q}."))].value_counts().reindex(order, fill_value=0) for s in scenarios})
    print(mat.T)
    print()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (q, order) in zip(axes.flat, orders.items()):
    rmap = {lvl: i for i, lvl in enumerate(order)}
    data = [df[next(c for c in df.columns if c.startswith(f"{s}.{q}."))].map(rmap).dropna().values for s in scenarios]
    ax.violinplot(data, positions=range(len(scenarios)), orientation="horizontal", showmeans=True, widths=0.9)
    ax.set_yticks(range(len(scenarios)), list(scenarios))
    ax.set_xticks(range(len(order)), order, rotation=45, ha="right")
    ax.set_title(f"{q} {labels[q]}")
    ax.invert_yaxis()
fig.tight_layout()
plt.show()

In [ ]:
q5 = [c for c in df.columns if c.startswith("Q5.")]
tags = {c: c.split("[")[1].strip("]") for c in q5}
ranks = pd.DataFrame({tags[c]: df[c].map(lambda v: int(str(v).split()[0])).values for c in q5}, index=df["Name"])

d = ranks.melt(var_name="scenario", value_name="val")
fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(ax=ax, data=d, x="val", y="scenario", order=list(ranks.columns), orient="h", cut=0, density_norm="count", bw_adjust=0.5, inner=None, color="C0", linewidth=0)
for coll in ax.collections:
    coll.set_alpha(0.3)
med = d.groupby("scenario")["val"].median().reindex(ranks.columns)
ax.scatter(med.values, range(len(ranks.columns)), marker="|", s=400, color="k", zorder=3)
ax.set_xticks([1, 2, 3, 4, 5], ["1 (least)", "2", "3", "4", "5 (most)"])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Q5 scenario ranking")
plt.show()

In [ ]:
order = list(ranks.columns)
dist = pd.DataFrame({s: ranks[s].value_counts().reindex(range(1, 6), fill_value=0) for s in order}).T
dist.columns = ["1 (least)", "2", "3", "4", "5 (most)"]
fig, ax = plt.subplots(figsize=(8, 5))
dist.plot.barh(stacked=True, colormap="RdYlGn_r", ax=ax)
ax.invert_yaxis()
ax.set_xlabel("participants")
ax.set_title("Q5 ranking")
ax.legend(title="rank", bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
q5e = "Briefly explain your answer to Q5 (< 50 words)"
quote = dict(zip(df["Name"], df[q5e]))
for name in ranks.index:
    print(f"===== {name} =====")
    print(quote[name])
    print()

In [ ]:
# --- Q5 heatmap, redesigned · scenarios as rows ------------------------
# Uses your existing `ranks` DataFrame (the one you passed to sns.heatmap
# as ranks.T): index = participant names, columns = labels like "S3-PR".
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

SHADES = ["#F3F0FA", "#DCD3F0", "#B7A4E3", "#8467CF", "#53389E"]  # ranks 1..5
TXT_ON = ["#4A4459", "#4A4459", "#3A3057", "white", "white"]
INK, MUTED = "#2A2440", "#8A85A0"
from matplotlib.colors import LinearSegmentedColormap
AVG_CMAP = LinearSegmentedColormap.from_list("q5", SHADES)  # continuous 1..5 shading

scenarios = list(ranks.columns)  # chronological S1..S5
people = list(ranks.index)
n_rows, n_cols = len(scenarios), len(people)

fig, ax = plt.subplots(figsize=(1.15 * n_cols + 4.1, 0.95 * n_rows + 2.0))
fig.subplots_adjust(left=0.13, right=0.9, top=0.76, bottom=0.04)
ax.set_xlim(0, n_cols + 1.5)
ax.set_ylim(n_rows, -0.85)  # inverted y; room above the grid for names
ax.axis("off")

g = 0.055  # gutter between tiles
for j, p in enumerate(people):  # column headers: first / last name stacked
    parts = str(p).split()
    name = parts[0] if len(parts) == 1 else parts[0] + "\n" + " ".join(parts[1:])
    ax.text(j + 0.5, -0.42, name, ha="center", va="center",
            fontsize=10, fontweight="semibold", color=INK, clip_on=False)
ax.text(n_cols + 0.85, -0.42, "mean", ha="center", va="center",
        fontsize=10, fontweight="semibold", color=INK, clip_on=False)

for i, s in enumerate(scenarios):
    code, dash, desc = str(s).partition("-")
    if not dash:
        code, desc = "", str(s)
    ax.text(-0.18, i + 0.40, desc.strip(), ha="right", va="center",
            fontsize=10.5, fontweight="semibold", color=INK, clip_on=False)
    if code:
        ax.text(-0.18, i + 0.68, code.strip(), ha="right", va="center",
                fontsize=8, color=MUTED, clip_on=False)
    for j, p in enumerate(people):
        v = int(ranks.loc[p, s])
        ax.add_patch(FancyBboxPatch(
            (j + g, i + g), 1 - 2 * g, 1 - 2 * g,
            boxstyle="round,pad=0,rounding_size=0.10",
            linewidth=0, facecolor=SHADES[v - 1], clip_on=False))
        ax.text(j + 0.5, i + 0.5, str(v), ha="center", va="center",
                fontsize=13, fontweight="bold", color=TXT_ON[v - 1])
    m = ranks[s].mean()
    xa = n_cols + 0.35
    ax.add_patch(FancyBboxPatch(
        (xa + g, i + g), 1 - 2 * g, 1 - 2 * g,
        boxstyle="round,pad=0,rounding_size=0.10",
        linewidth=0, facecolor=AVG_CMAP((m - 1) / 4), clip_on=False))
    ax.text(xa + 0.5, i + 0.5, f"{m:.1f}", ha="center", va="center",
            fontsize=12, fontweight="bold", color="white" if m >= 3.4 else INK)

fig.text(0.03, 0.94, "Q5 · Scenario significance rankings",
         fontsize=15, fontweight="bold", color=INK)
fig.text(0.03, 0.885,
         f"Each of {n_cols} participants force-ranked the {n_rows} scenarios · "
         "1 = least significant, 5 = most", fontsize=9.5, color=MUTED)
fig.text(0.03, 0.845, "Rows in scenario order (S1\u2013S5)", fontsize=9.5, color=MUTED)
plt.show()

In [ ]:
from scipy.stats import kendalltau

dfi = df.set_index("Name")
tag2s = {t: t.split("-")[0] for t in ranks.columns}
scodes = list(tag2s.values())  # scenario codes S1..S5 (don't rely on `scenarios`, it gets reassigned)
qcols = {q: {s: next(c for c in df.columns if c.startswith(f"{s}.{q}.")) for s in scodes} for q in orders}
rmaps = {q: {lvl: i for i, lvl in enumerate(order)} for q, order in orders.items()}

rows = {}
for name in ranks.index:
    rk = {tag2s[t]: ranks.loc[name, t] for t in ranks.columns}
    rvec = [rk[s] for s in scodes]
    row = {}
    for q in orders:
        qvec = [rmaps[q].get(dfi.loc[name, qcols[q][s]]) for s in scodes]
        row[labels[q]] = (float("nan") if None in qvec or len(set(qvec)) < 2
                          else kendalltau(rvec, qvec).correlation)  # tau-b, tie-corrected
    rows[name] = row

corr = pd.DataFrame(rows).T[[labels[q] for q in orders]]

# Aggregate = mean of per-person tau-b (unit = person, n=8; respects clustering
# and avoids the pseudoreplication/aggregation bias of pooling all 40 pairs).
corr.loc["mean (per-person)"] = corr.mean(axis=0)

# n=5 with heavy ties => enormous CIs, so 0.6 vs 0.8 is noise. Don't crown a single
# winner; bold every dimension within `margin` of each person's top tau-b instead.
def flag_near_top(r, margin=0.20):
    top = r.max()
    return ["font-weight:bold; border:1.5px solid #333" if pd.notna(v) and v >= top - margin else "" for v in r]

corr.style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1).format("{:+.2f}", na_rep="\u2014").apply(flag_near_top, axis=1)

In [ ]:
people = list(ranks.index)
qs = list(orders)
fig, axes = plt.subplots(len(people), len(qs), figsize=(2.9 * len(qs), 2.1 * len(people)), sharex=True)
for i, name in enumerate(people):
    rk = {tag2s[t]: ranks.loc[name, t] for t in ranks.columns}
    for j, q in enumerate(qs):
        ax = axes[i, j]
        order = orders[q]
        xs = [rk[s] for s in scodes]
        ys = [rmaps[q].get(dfi.loc[name, qcols[q][s]]) for s in scodes]
        ax.scatter(xs, ys, color=f"C{j}", zorder=3)
        for s, x, y in zip(scodes, xs, ys):
            if y is not None:
                ax.annotate(s, (x, y), fontsize=7, xytext=(3, 3), textcoords="offset points", color="0.4")
        pts = [(x, y) for x, y in zip(xs, ys) if y is not None]
        if len({y for _, y in pts}) > 1 and len({x for x, _ in pts}) > 1:
            X, Y = np.array([p[0] for p in pts]), np.array([p[1] for p in pts])
            m, b = np.polyfit(X, Y, 1)
            xr = np.array([1, 5])
            ax.plot(xr, m * xr + b, color="0.6", lw=1, zorder=1)
        ax.set_ylim(-0.5, len(order) - 0.5)
        ax.set_yticks(range(len(order)))
        ax.set_yticklabels(range(len(order)), fontsize=6)
        ax.set_xticks(range(1, 6))
        ax.grid(alpha=0.2)
        if i == 0:
            ax.set_title(f"{labels[q]}\n(low\u2192high \u2191)", fontsize=9)
    axes[i, 0].set_ylabel(name, fontsize=8, rotation=0, ha="right", va="center")
fig.supxlabel("Q5 rank (1 = least, 5 = most significant)")
fig.tight_layout()
plt.show()

In [ ]:
import itertools
from scipy.stats import kendalltau

dfi = df.set_index("Name")
tag2s = {t: t.split("-")[0] for t in ranks.columns}
scodes = list(tag2s.values())
qcols = {q: {s: next(c for c in df.columns if c.startswith(f"{s}.{q}.")) for s in scodes} for q in orders}
rmaps = {q: {lvl: i for i, lvl in enumerate(order)} for q, order in orders.items()}

def drivers(name):
    """Which single factor / equal-weight pair of Q1-Q4 (each oriented in this
    person's direction) best reproduces their Q5 ranking? n=5 => exploratory only."""
    M = pd.DataFrame({labels[q]: [rmaps[q].get(dfi.loc[name, qcols[q][s]]) for s in scodes] for q in orders}, index=scodes, dtype="float")
    rk = {tag2s[t]: ranks.loc[name, t] for t in ranks.columns}
    r = np.array([rk[s] for s in scodes], dtype=float)
    single, sign = {}, {}
    for c in M.columns:
        tau = kendalltau(r, M[c]).correlation if M[c].nunique() > 1 else np.nan
        single[c] = tau
        sign[c] = 0.0 if pd.isna(tau) else np.sign(tau)
    Z = M.apply(lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) else x * 0)
    Zo = (Z * pd.Series(sign)).fillna(0.0)
    cols = list(M.columns)
    best = (-2.0, None)  # each single factor + each equal-weight pair
    for cmb in [(c,) for c in cols] + list(itertools.combinations(cols, 2)):
        comp = Zo[list(cmb)].mean(axis=1).values
        if len(set(np.round(comp, 9))) < 2:
            continue
        tau = kendalltau(r, comp).correlation
        if tau is not None and tau > best[0]:
            best = (tau, cmb)
    return single, sign, best

rows = {}
for name in ranks.index:
    single, sign, (btau, bmodel) = drivers(name)
    top = max((abs(v) for v in single.values() if pd.notna(v)), default=np.nan)
    model = " + ".join(f"{c}{'' if sign[c] >= 0 else '(\u2212)'}" for c in bmodel)
    rows[name] = {**single, "best model": model, "model \u03c4": btau, "lift": btau - top,
                  "type": "single" if len(bmodel) == 1 or btau - top < 0.10 else "combo"}

drv = pd.DataFrame(rows).T
num = list(labels.values()) + ["model \u03c4", "lift"]
drv.style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1, subset=list(labels.values())).format("{:+.2f}", subset=num)

In [ ]:
q6 = next(c for c in df.columns if c.startswith("Q6."))
q6e = next(c for c in df.columns if c.startswith("Briefly explain your answer to Q6"))

# full option list from the survey (most -> least concern; most on top after invert)
order = ["Among my top 2-3 concerns", "Toward the top of my list", "Toward the middle of my list",
         "Well down my list", "Nowhere on my list"]
counts = df[q6].value_counts().reindex(order, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(range(len(counts)), counts.values, color=["C0" if v else "0.85" for v in counts.values])
ax.set_yticks(range(len(counts)), counts.index)
ax.invert_yaxis()
ax.set_xlabel("participants")
ax.set_title("Q6 \u00b7 where AI-persuasion risk sits among concerns")
for i, v in enumerate(counts.values):
    ax.text(v + 0.03, i, int(v), va="center", color="0.6" if v == 0 else "black")
plt.show()

for lvl in order:
    sub = df[df[q6] == lvl]
    if sub.empty:
        continue
    print(f"===== {lvl} ({len(sub)}) =====")
    for name, e in zip(sub["Name"], sub[q6e]):
        print(f"  {name}: {e}")
    print()

In [ ]:
q7 = next(c for c in df.columns if c.startswith("Q7."))
q7e = next(c for c in df.columns if c.startswith("Briefly explain your answer to Q7"))
q7labels = {f"Q{n}": lab for n, lab in zip(range(1, 7), ["realism", "persuasion", "frequency", "impact", "marginal", "vs others"])}

sel = pd.DataFrame({f"Q{n}": df[q7].fillna("").str.contains(fr"Q{n}\.").values for n in range(1, 7)}, index=df["Name"]).astype(int)
cols = [f"{k} {v}" for k, v in q7labels.items()]

fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1, 1.4]})
n = len(sel)
counts = sel.sum()
axes[0].bar(range(6), [n] * 6, color="0.9", zorder=0)  # faint full-height = all participants
axes[0].bar(range(6), counts.values, color="C0", zorder=1)
axes[0].axhline(n, color="0.5", ls="--", lw=1)
axes[0].text(5.6, n, f"n={n}", va="center", ha="left", color="0.5", fontsize=9, clip_on=False)
axes[0].set_xticks(range(6), cols, rotation=45, ha="right")
axes[0].set_yticks(range(n + 1))
axes[0].set_ylim(0, n)
axes[0].set_ylabel("respondents")
axes[0].set_title("Q7 \u00b7 answers seen as timeframe-dependent")
for i, v in enumerate(counts.values):
    axes[0].text(i, v - 0.3, f"{int(v)}/{n}", ha="center", va="top", color="white", fontweight="bold")

ax = axes[1]
ax.imshow(sel.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(6), cols, rotation=45, ha="right")
ax.set_yticks(range(len(sel)), sel.index)
for i in range(len(sel)):
    for j in range(6):
        if sel.values[i, j]:
            ax.text(j, i, "\u2713", ha="center", va="center", color="white", fontsize=11)
ax.set_title("who flagged which (\u2713 = timeframe-dependent)")
fig.tight_layout()
plt.show()

for name, e in zip(df["Name"], df[q7e]):
    flagged = [q7labels[f"Q{n}"] for n in range(1, 7) if sel.loc[name, f"Q{n}"]]
    print(f"===== {name} ({len(flagged)}) =====")
    print(f"  flagged: {', '.join(flagged) or 'none'}")
    print(f"  {e}")
    print()

In [ ]:
q8 = [c for c in df.columns if c.startswith("Q8")]
q8e = next(c for c in df.columns if c.startswith("Briefly explain your answer to Q8"))
q8labels = ["Q1 realism", "Q2 persuasion", "Q3 frequency", "Q4 impact", "Q5 marginal", "Q6 vs others"]
corder = ["Very Low", "Low", "Medium", "High", "Very High"]
cmap_conf = {lvl: i for i, lvl in enumerate(corder)}
abbr = {0: "VL", 1: "L", 2: "M", 3: "H", 4: "VH"}

conf = pd.DataFrame({lab: df[c].map(cmap_conf).values for lab, c in zip(q8labels, q8)}, index=df["Name"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1, 1.5]})

meds = conf.median().sort_values()  # lowest = group finds it hardest to estimate
axes[0].barh(range(len(meds)), meds.values, color="C0")
axes[0].set_yticks(range(len(meds)), meds.index)
axes[0].invert_yaxis()
axes[0].set_xlim(0, 4)
axes[0].set_xticks(range(5), corder, rotation=45, ha="right")
axes[0].set_title("Q8 \u00b7 median confidence by question")
for i, v in enumerate(meds.values):
    axes[0].text(v + 0.06, i, f"{v:.1f}", va="center")

ax = axes[1]
ax.imshow(conf.values, cmap="RdYlGn", vmin=0, vmax=4, aspect="auto")
ax.set_xticks(range(len(q8labels)), q8labels, rotation=45, ha="right")
ax.set_yticks(range(len(conf)), conf.index)
for i in range(len(conf)):
    for j in range(len(q8labels)):
        ax.text(j, i, abbr[conf.values[i, j]], ha="center", va="center", fontsize=9, color="0.15")
ax.set_title("confidence by participant")
fig.tight_layout()
plt.show()

for name, e in zip(df["Name"], df[q8e]):
    print(f"===== {name} =====")
    print(f"  {e}")
    print()

In [ ]:
extra = next(c for c in df.columns if c.startswith("Anything else"))
for name, e in zip(df["Name"], df[extra]):
    if pd.isna(e):
        continue
    print(f"===== {name} =====")
    print(f"  {e}")
    print()